In [3]:
import numpy as np
import pandas as pd
import scipy

from IPython.display import clear_output

import re
from pathlib import Path
import requests
import random
import time
import scipy
from bs4 import BeautifulSoup
import gc
import json
import glob

from tqdm import tqdm
import networkx as nx

In [27]:
url_df = pd.read_csv('../data/261028_clean_urls.csv')

In [28]:
url_df[url_df['global_uid'] == 15636]

,service,global_uid,service_uid,url,clean_url,n_pred_services
29641,cosmos,15636,cosmos_4025,http://www.indianjournals.com,indianjournals.com,2
29642,drji,15636,drji_0975-5454,http://www.indianjournals.com/ijor.aspx?target...,indianjournals.com,2


In [29]:
uids = url_df['global_uid'].unique()

In [42]:
np.random.seed(0)
select_uids = np.random.choice(uids, 201)

In [43]:
sel_df = url_df[url_df['global_uid'].isin(select_uids)].sort_values('global_uid').drop_duplicates(['global_uid', 'url'])

In [45]:
sel_df

,service,global_uid,service_uid,url,clean_url,n_pred_services
725,sjif,307,sjif_18027,http://empyreal.co.in/journal-current-issue.html,empyreal.co.in,5
726,esji,307,esji_1786.0,http://www.empyreal.co.in/journal.html,empyreal.co.in,5
728,cosmos,307,cosmos_245,http://www.empyreal.co.in,empyreal.co.in,5
729,drji,307,drji_2322-0899,www.empyreal.co.in/journal.html,empyreal.co.in,5
1063,sjif,469,sjif_20428,http://cmro.in/index.php/jcmro,cmro.in,2
...,...,...,...,...,...,...
34887,drji,20587,drji_2349-7165,https://unnayan.ipsacademy.org/,unnayan.ipsacademy.org,1
34962,drji,20662,drji_1970-1950,https://www.mediaevalsophia.net/it/,mediaevalsophia.net,1
35053,drji,20753,drji_2346-9420,https://revistasad.com/index.php/diabetes/,revistasad.com,1
35073,drji,20773,drji_3005-8449,https://irabcs.com,irabcs.com,1


In [46]:
sel_df.to_csv('../data/261030_select_journals_inspect_unfilled.csv', index=False)

In [109]:
sel_df = pd.read_csv('../data/261030_select_journals_inspect_filled.csv')

In [110]:
sel_df.groupby('global_uid')['notes_on_indexing'].nunique().value_counts()

1    200
Name: notes_on_indexing, dtype: int64

In [111]:
sel_df['global_uid'].nunique()

200

In [112]:
sel_df = sel_df.drop_duplicates(['global_uid', 'notes_on_indexing']).copy()

In [113]:
sel_df['clean_notes'] = sel_df['notes_on_indexing']

In [114]:
for key in ['sjif', 'esji', 'drji', 'ici', 'ifj', 'ifsij', 'iifs', 'cosmos']:
    sel_df.loc[sel_df['clean_notes'].str.contains(key), 'clean_notes'] = 'pred'

In [115]:
permit = ['pred', "won't load", "won't load (404)", 'none', 'unable to locate indexing page', 'no longer journal website']

In [116]:
sel_df.loc[~sel_df['clean_notes'].isin(permit), 'clean_notes'] = 'none'

In [117]:
sel_df['clean_notes'].value_counts()

none                              62
pred                              56
won't load                        38
unable to locate indexing page    22
no longer journal website         14
won't load (404)                   8
Name: clean_notes, dtype: int64

In [118]:
sel_df.sample(10)

,service,global_uid,service_uid,url,clean_url,n_pred_services,notes_on_indexing,indexing_page_link,other_notes,clean_notes
13,sjif,1134,sjif_20608,http://www.ijoes.in,ijoes.in,2,"sjif, drji",https://www.ijoes.in/,NaN,pred
210,esji,14384,esji_8629.0,https://rjsaonline.org/index.php/EcoBiotics,rjsaonline.org,2,"esji, cosmos, sjif",https://rjsaonline.org/index.php/EcoBiotics/In...,NaN,pred
291,drji,20773,drji_3005-8449,https://irabcs.com,irabcs.com,1,drji,https://irjpl.org/irabcs/IndexingAndAbstracting,NaN,pred
113,drji,7368,drji_2786-4936,http://ejsit-journal.com/index.php/ejsit,ejsit-journal.com,4,won't load (404),NaN,NaN,won't load (404)
256,drji,17356,drji_2686-3219,https://teknokom.unwir.ac.id/index.php/teknokom,teknokom.unwir.ac.id,1,none,https://www.agriscigroup.us/journals/internati...,NaN,none
246,cosmos,16314,cosmos_8029,http://spmj.keyem.marmara.edu.tr,spmj.keyem.marmara.edu.tr,1,doaj,https://spmj-keyem.marmara.edu.tr/strategic-pu...,NaN,none
281,drji,19664,drji_2654-5586,http://ejurnal.budiutomomalang.ac.id/index.php...,ejurnal.budiutomomalang.ac.id,1,won't load,NaN,NaN,won't load
182,esji,12380,esji_5465.0,http://www.pphmj.com/journals/ijmet.htm,pphmj.com,1,esji,https://www.pphmj.com/journals/ijmet_reviews.htm,NaN,pred
283,drji,19821,drji_2519-4518,https://mhpi.chnu.edu.ua,mhpi.chnu.edu.ua,1,unable to locate indexing page,NaN,NaN,unable to locate indexing page
117,sjif,7880,sjif_22185,http://jmss.a2zjournals.com/index.php/mss,jmss.a2zjournals.com,1,none,https://jmss.a2zjournals.com/index.php/mss/ind...,NaN,none


In [119]:
sel_df[sel_df['clean_notes'] == 'pred'].sort_values('n_pred_services')

,service,global_uid,service_uid,url,clean_url,n_pred_services,notes_on_indexing,indexing_page_link,other_notes,clean_notes
131,sjif,8470,sjif_22376,http://aipublications.com/ijebm/,aipublications.com,1,drji,https://aipublications.com/ijebm/,NaN,pred
158,esji,10204,esji_1860.0,http://www.worldnewsnaturalsciences.com/,worldnewsnaturalsciences.com,1,esji,https://www.worldnewsnaturalsciences.com/abstr...,NaN,pred
182,esji,12380,esji_5465.0,http://www.pphmj.com/journals/ijmet.htm,pphmj.com,1,esji,https://www.pphmj.com/journals/ijmet_reviews.htm,NaN,pred
195,esji,13605,esji_7530.0,https://www.envsciarch.com/,envsciarch.com,1,"wos, esji",NaN,NaN,pred
202,esji,13880,esji_7939.0,https://journal.publiseditorial.com/index.php/...,journal.publiseditorial.com,1,esji,https://journal.publiseditorial.com/index.php/...,NaN,pred
204,esji,14132,esji_8328.0,https://accscience.com/journal/BH,accscience.com,1,esji,https://accscience.com/journal/BH/about/indexing,NaN,pred
223,iifs,15057,iifs_2429,https://ajaccm.com/journal/index.php/ajaccm,ajaccm.com,1,iifs,https://ajaccm.com/journal/index.php/ajaccm/in...,NaN,pred
247,cosmos,16337,cosmos_8195,http://irispublishers.com,irispublishers.com,1,"sjif, esji, cosmos, drji",NaN,NaN,pred
138,sjif,8755,sjif_24689,http://journals.stmjournals.com/ijv/,journals.stmjournals.com,1,sjif,https://journals.stmjournals.com/ijv/,NaN,pred
250,drji,16937,drji_2574-2574,http://www.heraldopenaccess.us/journals/Reprod...,heraldopenaccess.us,1,drji,http://www.heraldopenaccess.us/journals/Reprod...,NaN,pred


In [120]:
one_row_per_journal_df = pd.read_csv('../data/261028_one_row_per_journal.csv')

In [121]:
one_row_per_journal_df[one_row_per_journal_df['global_uid'] == 18692].values

array([[18692, 'Surgery Clinics Journal',
        'http://www.medtextpublications.com/surgery-clinics-journal-home.php',
        '2687-6973', 'drji', 1.0]], dtype=object)

In [122]:
one_row_per_journal_df[one_row_per_journal_df['titles'].str.lower().str.contains('yoga')]

,global_uid,titles,urls,issns,services,n_pred_services
809,5034,International Research Journal of Ayurveda & Y...,"http://irjay.com, http://www.irjay.com",2581-785X,"cosmos, iifs, sjif, esji",4.0
1306,10301,"Indian Journal of Youth and Adolescent Health,...",http://medical.adrpublications.in/index.php/Jo...,2394-6547,"ifsij, drji, esji, cosmos",4.0
2052,3496,International Journal of Physical Education Yo...,"http://snpublication.com/journal/ijeppeye, htt...",2349-6312,"ifsij, sjif, iifs",3.0
3426,1731,International Journal of Adapted Physical Educ...,"http://www.ijapey.info/, http://www.ijapey.info","2229-4821, 2455-8958","cosmos, sjif, esji",3.0
3724,2526,"INDIAN JOURNAL OF ANCIENT MEDICINE AND YOGA, I...","http://rfppl.co.in, http://rfppl.co.in/about_j...","0974-6986, 0974-6994","esji, sjif, cosmos",3.0
4882,7499,"Journal of Advances in Ayurveda, Yoga, Homeopa...","http://hbrppublication, http://hbrppublication...",3048-4723,"sjif, cosmos",2.0
6800,13738,Journal of Yoga & Physical Therapy,https://www.longdom.org/yoga-physical-therapy....,2157-7595,"esji, cosmos",2.0
8710,3472,"Research & Reviews: Journal of Ayush, Journal ...",http://www.stmjournals.com/med/index.php?journ...,"2278-2214, 2321-6484","sjif, iifs",2.0
8995,15027,Aathiyoga Indian Journal of Ancient Medicine a...,https://aathiyoga.in,3048-9822,iifs,1.0
9264,16615,National Journal of Medical Research & Yoga Sc...,http://njmrys.com,2395-2911,cosmos,1.0
